## Homework: AI Orchestration with Kestra

ATTENTION: At the end of the submission form, you will be required to include a link to your GitHub repository or other public code-hosting site. This repository should contain your code for solving the homework. If your solution includes code that is not in file format, please include these directly in the README file of your repository.

> It's possible your answers won't match exactly. If so, select the closest one

## Prerequisites

Before starting this homework, ensure you have:

1. Completed the [Module 3 lessons](../../../03-orchestration/README.md) — the questions reference flows and concepts covered there
2. Kestra running locally with API keys configured (see the [Setup](../../../03-orchestration/lessons/03-setup.md) lesson) -- this includes the Gemini API key, which is also required for the AI Copilot
3. Imported all flows from the `03-orchestration/flows/` directory (covered in the Setup lesson)

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

## Question 1: Context Engineering

Try the following experiment:

1. Open ChatGPT in a private browser window: https://chatgpt.com
2. Enter this prompt: "Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"
3. Then, use Kestra's AI Copilot with the same prompt

After trying the same prompt in ChatGPT vs Kestra's AI Copilot, what is the primary reason AI Copilot generates better Kestra flows?

- AI Copilot uses a more powerful model
- AI Copilot has access to current Kestra plugin documentation
- AI Copilot uses more tokens
- AI Copilot has internet access

### ChatGPT:

```
id: nyc_taxi_csv_to_bigquery
namespace: company.team

variables:
  project_id: your-gcp-project
  dataset: nyc_taxi
  table: yellow_tripdata
  bucket: your-gcs-bucket

tasks:
  - id: download_csv
    type: io.kestra.plugin.core.http.Download
    uri: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2023-01.csv.gz

  - id: decompress
    type: io.kestra.plugin.compress.ArchiveDecompress
    from: "{{ outputs.download_csv.uri }}"

  - id: upload_to_gcs
    type: io.kestra.plugin.gcp.gcs.Upload
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT') }}"
    from: "{{ outputs.decompress.files[0] }}"
    bucket: "{{ vars.bucket }}"
    name: yellow_tripdata_2023-01.csv

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.LoadFromGcs
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT') }}"
    projectId: "{{ vars.project_id }}"
    dataset: "{{ vars.dataset }}"
    table: "{{ vars.table }}"
    from:
      - "gs://{{ vars.bucket }}/yellow_tripdata_2023-01.csv"
    format: CSV
    autodetect: true
    skipLeadingRows: 1
    writeDisposition: WRITE_TRUNCATE
    createDisposition: CREATE_IF_NEEDED
```

### Kestra

```
id: load_nyc_taxi_data_to_bigquery
namespace: dev

tasks:
  - id: download_taxi_data
    type: io.kestra.plugin.core.http.Download
    uri: "https://huggingface.co/datasets/kestra/datasets/raw/main/csv/orders.csv" # Replace with the actual URL of your NYC taxi data CSV file
    saveAs: "nyc_taxi_data.csv"

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    from: "{{ outputs.download_taxi_data.uri }}"
    destinationTable: "your_project.your_dataset.nyc_taxi_data" # Replace with your BigQuery destination table (e.g., project_id.dataset_id.table_id)
    format: CSV
    csvOptions:
      skipLeadingRows: 1 # Set to the number of header rows in your CSV file, if any
      allowJaggedRows: true
      allowQuotedNewLines: true
      fieldDelimiter: ","
    # projectId: "your-gcp-project-id" # Uncomment and replace with your GCP Project ID if not using a default configured one


```

### Answer: AI Copilot has access to current Kestra plugin documentation

## Question 2: RAG vs No RAG

Run both `1_chat_without_rag.yaml` and `2_chat_with_rag.yaml` in the Kestra UI. Read the execution logs for each.

The non-RAG response about Kestra 1.1 features is best described as:

- Accurate and specific, matching the actual release notes
- Vague, generic, or fabricated — the model guesses from training data
- Empty — the model refuses to answer without context
- Identical to the RAG version



### Execution log for the non-RAG flow:
INFO 2026-07-23T08:57:46.311754Z ❌ Response WITHOUT RAG (no retrieved context):
Kestra 1.1 introduced several exciting features that significantly enhanced its capabilities. Here are 5 major features with brief descriptions:

1.  **Event-Driven Flows (Triggers):** This was a major leap forward, allowing Kestra workflows to be automatically initiated by external events. Instead of just scheduled runs, you could now set up triggers for things like new files in an S3 bucket, messages on a Kafka topic, or custom webhook calls. This feature transformed Kestra into a powerful event-driven automation platform, enabling real-time reactions to data and system changes.

2.  **Kestra Command Line Interface (CLI):** Kestra 1.1 introduced a dedicated CLI tool. This allowed users to interact with Kestra instances from their terminal, providing functionalities like deploying flows, managing executions, viewing logs, and more. The CLI significantly improved developer experience and enabled easier integration with CI/CD pipelines and automated deployments.

3.  **New Namespace Page & Namespace-Specific RBAC:** This release brought a redesigned and more powerful Namespace page within the UI. Crucially, it introduced **Namespace-Specific Role-Based Access Control (RBAC)**. This meant administrators could define granular permissions for users and groups at the namespace level, allowing different teams to manage their flows independently without interfering with others, and enhancing security and governance for larger organizations.

4.  **Templating & Reusability Improvements (New UI for Templates):** Kestra 1.1 continued to focus on making flows more modular and reusable. While templating existed, this release brought **significant UI improvements for managing and visualizing templates**. This made it easier for users to define, discover, and reuse common logic across multiple flows, reducing duplication and promoting consistency in their Kestra projects.

5.  **Enhanced Worker Group Management & Isolation:** This feature provided more robust control over how tasks are distributed and executed. Users could define and assign tasks to specific "worker groups," allowing for better resource isolation and prioritization. For example, CPU-intensive tasks could be assigned to a worker group on powerful machines, while I/O-bound tasks could go to another. This improved performance, stability, and resource utilization, especially in complex and demanding environments.

🤔 Did you notice that this response seems to be:
- Incorrect?
- Vague/generic?
- Listing features that haven't been added in exactly this version but rather a long time ago?

👉 This is why context matters! Run `2_chat_with_rag.yaml` to see the accurate, context-grounded response.


### Execution log for the RAG flow:
INFO 2026-07-23T09:01:04.860827Z ✅ RAG Response (with retrieved context):
Kestra 1.1 introduced several major features, including:

1.  **New Filters**: The UI filters were completely redesigned for improved usability, offering explicit filter options, single-click resets, the ability to save frequently used filter combinations, and customizable table columns.
2.  **No-Code Dashboard Editor**: This feature allows users to create and edit dashboards using a no-code, multi-panel editor directly from the UI, similar to the existing no-code flow editor. Users can design dashboards without writing YAML.
3.  **Human Task**: (Enterprise Edition) This new task enables human-in-the-loop workflows, allowing executions to pause and require manual approval from specific users or groups before proceeding.
4.  **Multi-Agent AI Systems**: AI agents can now use other AI agents as tools, facilitating sophisticated multi-agent orchestration workflows where a primary agent can delegate subtasks to specialized expert agents.
5.  **Fix with AI**: When tasks fail, Kestra 1.1 provides AI-powered suggestions to help users quickly diagnose and resolve issues, speeding up troubleshooting.

🎉 Note that this response is detailed, accurate, and grounded in the actual release documentation. Compare this with the output from 1_chat_without_rag.yaml!


### Answer: Vague, generic, or fabricated — the model guesses from training data

## Question 3: Token usage — short summary

Run `4_simple_agent.yaml` with `summary_length = short` (leave the other inputs as defaults).

Open the execution logs and find the token usage logged by the `log_token_usage` task.

What is the approximate **output** token count for `multilingual_agent`?

- 5-15 tokens
- 60-100 tokens
- 200-400 tokens
- 500+ tokens



### Asnwer: 67 tokens, so 60-100 tokens

## Question 4: Token usage — long summary

Run `4_simple_agent.yaml` again with `summary_length = long`.

Compare the `multilingual_agent` output token count to your result from Question 3. Roughly how many times more output tokens does the long summary use?

- About the same (within 20%)
- 2-5x more
- 10-20x more
- 50x more



### Answer: 184 tokens, so compared to 67 tokens in Q3, the answer is 2-5x more

## Question 5: Modifying a flow

Open `4_simple_agent.yaml` in the Kestra flow editor. Find the `english_brevity` task and change its prompt from asking for exactly **1 sentence** to asking for exactly **3 sentences**.

Save the flow, then run it with `summary_length = long`.

Compare the `english_brevity` output token count to the original 1-sentence version (also with `summary_length = long`). How do they compare?

- About the same (within 20%)
- 2-4x more
- 5-10x more
- 10x+ more



### Answer: 2-4x times (I did it yesterday, but failed to save, and right now I exhausted resource limit)

## Question 6: Best Practices

Based on what you learned in this module, for production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, workflows in highly regulated industries), which approach is most appropriate?

- Always use AI agents for maximum flexibility and adaptation
- Use traditional task-based workflows for predictability and auditability
- Use only RAG without agents for better performance
- Use web search tools exclusively to ensure current data

### Answer: Use traditional task-based workflows for predictability and auditability